# Adhan-nano pretraining on a free Colab T4

9 layers · 256 d_model · 8 query heads / 2 KV heads (GQA) · dropout 0.1 · pure RoPE · **16,740,096 params**

Runtime → Change runtime type → **T4 GPU**. Full recipe and caveats: `docs/COLAB_T4.md`.

Budget: the whole 293.5M-token archive is ~3.3–5.4 h at the expected 15–25k tok/s.

In [ ]:
# 1) JAX with CUDA 12 (the preinstalled jax is CPU-only; T4 = sm_75, fp16 tensor cores)
!pip -q install "jax[cuda12]" flax optax orbax-checkpoint mlflow "tokenizers>=0.15"
import jax
print(jax.devices())  # expect [CudaDevice(id=0)]
assert any(d.platform == "gpu" for d in jax.devices()), "No GPU: Runtime -> Change runtime type -> T4 GPU"

In [ ]:
# 2) repo + packed shards
%cd /content
!git clone -q https://github.com/yazhi-lem/adhan.git
%cd /content/adhan
!pip -q install -e .

# Shards: either pull the published dataset, or upload your own train.bin/val.bin
HF_SHARDS = ""  # e.g. "<user>/adhan-tamil-shards"
import os, pathlib
dst = pathlib.Path("data/final/tamil_slm_v2_s512"); dst.mkdir(parents=True, exist_ok=True)
if HF_SHARDS:
    from huggingface_hub import snapshot_download
    snapshot_download(HF_SHARDS, repo_type="dataset", local_dir=str(dst))
else:
    print("set HF_SHARDS, or upload train.bin/val.bin/*.manifest.json/vocab.json + tokenizer.json into", dst)

In [ ]:
# 3) keep checkpoints across preemptions (strongly recommended)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/adhan", exist_ok=True)
    os.symlink("/content/drive/MyDrive/adhan", "checkpoints", target_is_directory=True)
    print("checkpoints -> Drive")
except Exception as e:
    print("no Drive (checkpoints will die with the session):", e)

In [ ]:
# 4) wiring gate: loss must collapse (target <50% of start) in ~3 min
!python -m adhan_slm.training.train_jax \
    --config src/adhan_slm/configs/adhan_slm_nano_t4.yaml --overfit-batch --device gpu

In [ ]:
# 5) the run — rerun this exact cell after a preemption; it resumes from the last
#    checkpoint and continues the global step budget.
!python -m adhan_slm.training.train_jax \
    --config src/adhan_slm/configs/adhan_slm_nano_t4.yaml --device gpu

In [ ]:
# 6) what the run produced, straight from MLflow (no server needed)
import mlflow
from mlflow.tracking import MlflowClient
mlflow.set_tracking_uri("sqlite:///mlflow.db")
c = MlflowClient(); e = c.get_experiment_by_name("adhan-nano-t4")
r = c.search_runs([e.experiment_id], order_by=["attributes.start_time DESC"], max_results=1)[0]
m = r.data.metrics
print(r.info.run_id[:8], r.info.status)
print({k: round(v, 3) for k, v in sorted(m.items())})
print("val_loss best:", c.get_metric_history(r.info.run_id, "val_loss")[-1].value if "val_loss" in m else None)

## Notes

* **fp16, not bf16.** T4 is Turing: no bf16 units. Master weights and the output head stay fp32, which is what keeps fp16 stable here. NaN → switch `model.dtype` to `float32`.
* **Throughput check.** Under ~15k tok/s usually means: no GPU reported, `bfloat16` still in the config, or `batch_size` left at 8 (launch-overhead-bound).
* **After training:** copy the best checkpoint out of the VM (`checkpoints/adhan-nano-t4`), then run `run_eval` → `quantize_model.py` → `run_api_server.py` on the dev box.